# 11. Visualizing Comparisons & Video

Notebook 10 plotted one sequence at a time. This notebook covers comparing multiple sequences — feature
relationships, distributions across detectors/raters, and full label-sequence scarfplots — and rendering gaze data
as a video.

In [1]:
import numpy as np
import peyes
import _helpers

d = _helpers.load_example_trial()
ground_truth_labels = d["raters"]["RA"]
detector = peyes.create_detector("engbert", missing_value=np.nan, min_event_duration=4, pad_blinks_time=0)
prediction_labels, _ = detector.detect(
    t=d["t"], x=d["x"], y=d["y"], pixel_size_cm=d["pixel_size"], viewer_distance_cm=d["viewer_distance"],
)
events = peyes.create_events(
    labels=prediction_labels, t=d["t"], x=d["x"], y=d["y"], pupil=d["pupil"],
    pixel_size=d["pixel_size"], viewer_distance=d["viewer_distance"],
)
fixations = [e for e in events if e.label == peyes.parse_label("fixation")]
saccades = [e for e in events if e.label == peyes.parse_label("saccade")]

## Feature relationships

`feature_relationship` scatters two features of one event sequence against each other, with an optional trendline
(`trendline="overall"` fits one line across all points):

In [2]:
fig, trend = peyes.visualize.feature_relationship(
    events, x_feature="duration", y_feature="amplitude", trendline="overall",
)
fig

`main_sequence` is a named special case of this: saccade amplitude vs. duration (or peak velocity), the
classic "main sequence" relationship in the oculomotor literature:

In [3]:
fig, _ = peyes.visualize.main_sequence(saccades, y_feature="duration")
fig

## Comparing distributions across sequences

`feature_comparison` draws a ridge plot of one feature's distribution across several event sequences side by side —
here, fixations vs. saccades, but this is the same call you'd use to compare the same label across detectors:

In [4]:
peyes.visualize.feature_comparison("duration", fixations, saccades)

## Comparing whole label sequences: scarfplots

`scarfplot_comparison_figure` stacks color-coded per-sample label strips for several sequences, aligned in time —
a quick visual gestalt for "where do these sequences agree or disagree":

In [5]:
peyes.visualize.scarfplot_comparison_figure(
    d["t"], ground_truth_labels, prediction_labels, names=["RA (human)", "Engbert"],
)

## Exporting a video

`create_video` renders the moving gaze point, colored by event label, frame by frame into an `.mp4` file (fps is
inferred from the sampling rate); `create_frames` returns the raw frame array sequence without encoding a video, if
you want to process frames yourself. Blink samples have no gaze coordinate (`x`/`y` are `NaN`), so we render the
first 1000 samples of this trial, before its one blink, as a short demo clip:

In [6]:
import os

n = 1000
output_path = os.path.join("data", "example_trial.mp4")
peyes.visualize.create_video(
    d["t"][:n], d["x"][:n], d["y"][:n], prediction_labels[:n], output_path=output_path, resolution=(1920, 1080),
)
print(f"{os.path.getsize(output_path) / 1024:.0f} KB written to {output_path}")

2841 KB written to data\example_trial.mp4


## What's next

This was the last notebook in the guide. From here, `help(peyes)` and each module's docstrings (e.g.
`help(peyes.event_metrics)`) are the reference for the full API surface — every function used across these 11
notebooks, plus a few narrow helpers not covered here.